# 문제 1-2 : 2단계 Multi Chain 만들기 - 여행지 정보 시스템

여행지(도시/국가)를 입력하면
1. **1단계 체인**: 대표 관광 명소 1가지를 추천하고
2. **2단계 체인**: 추천받은 명소의 상세 정보(역사, 특징, 방문 팁)를 알려줍니다.

- `ChatPromptTemplate`으로 `system` / `user` 메시지 지정
- 두 체인을 LCEL로 연결
- 각 단계의 결과를 모두 출력

## 1. 환경 설정

In [4]:
from dotenv import load_dotenv
load_dotenv()

import os
print("OpenAI Key:", "OK" if os.getenv("OPENAI_API_KEY") else "없음")

OpenAI Key: OK


In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
output_parser = StrOutputParser()

## 2. 1단계 체인 - 대표 명소 1가지 추천

In [6]:
prompt1 = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 전문가입니다. 질문한 여행지의 가장 대표적인 관광 명소 1곳의 이름만 한국어로 답하세요. 설명이나 문장부호 없이 명소 이름만 출력하세요."),
    ("user", "여행지: {destination}"),
])

chain1 = prompt1 | llm | output_parser

# 1단계만 단독 테스트
print(chain1.invoke({"destination": "로마"}))

콜로세움


## 3. 2단계 체인 - 명소 상세 정보

In [7]:
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 가이드입니다. 주어진 관광 명소에 대해 한국어로 상세히 소개하세요."),
    ("user", """명소: {attraction}

아래 형식으로 답변하세요.

명소: (명소 이름)
역사:
(역사 설명)
특징:
(특징 설명)
방문 팁:
(방문 팁)"""),
])

chain2 = prompt2 | llm | output_parser

# 2단계만 단독 테스트
print(chain2.invoke({"attraction": "콜로세움"}))

명소: 콜로세움  
역사:  
콜로세움은 이탈리아 로마에 위치한 고대 원형 경기장으로, 서기 70년에서 80년 사이에 건설되었습니다. 로마 제국의 황제 베스파시아누스에 의해 시작되어 그의 아들 티투스에 의해 완공되었습니다. 이곳은 고대 로마의 대중 오락을 위한 중심지로, 검투사 경기, 야생 동물 사냥, 그리고 다양한 공연이 열렸습니다. 콜로세움은 로마의 상징적인 건축물로, 그 당시의 기술과 예술을 보여주는 중요한 유산입니다. 

특징:  
콜로세움은 직경 약 188미터, 높이 약 57미터의 거대한 구조물로, 최대 50,000명에서 80,000명까지 수용할 수 있었습니다. 이 경기장은 80개의 출입구와 여러 층으로 이루어져 있으며, 그 내부는 복잡한 통로와 계단으로 구성되어 있습니다. 또한, 콜로세움의 외관은 고대 로마 건축의 대표적인 예로, 아치형 구조와 기둥 장식이 특징입니다. 현재는 유네스코 세계문화유산으로 지정되어 있으며, 로마의 역사와 문화를 대표하는 중요한 관광 명소입니다.

방문 팁:  
콜로세움을 방문할 때는 사전 예약을 추천합니다. 특히 성수기에는 긴 대기줄이 생길 수 있으므로, 온라인으로 미리 티켓을 구매하면 시간을 절약할 수 있습니다. 또한, 가이드 투어를 이용하면 콜로세움의 역사와 구조에 대한 깊이 있는 설명을 들을 수 있어 더욱 풍성한 경험을 할 수 있습니다. 방문 시에는 편안한 신발을 착용하고, 충분한 물과 햇볕을 피할 수 있는 모자를 챙기는 것이 좋습니다. 저녁 시간대에 방문하면 아름다운 조명 아래에서 콜로세움의 또 다른 매력을 느낄 수 있습니다.


## 4. 두 체인을 LCEL로 연결

### 4-1. 기본 연결

`{"attraction": chain1}` 딕셔너리가 1단계 결과를 `attraction` 키에 담아 2단계 체인에 넘겨줍니다.
최종 결과(2단계)만 반환되므로 중간 결과는 보이지 않습니다.

In [8]:
full_chain = {"attraction": chain1} | chain2

print(full_chain.invoke({"destination": "도쿄"}))

명소: 도쿄 타워

역사:
도쿄 타워는 1958년에 완공된 일본의 상징적인 구조물로, 당시 일본의 경제 성장과 기술 발전을 상징하는 아이콘으로 자리 잡았습니다. 높이는 333미터로, 파리의 에펠탑을 모델로 하여 건설되었습니다. 도쿄 타워는 방송 송신탑으로서의 기능을 가지고 있으며, 일본 전역에 TV 및 라디오 신호를 송출하는 중요한 역할을 하고 있습니다. 1980년대와 2000년대 초반에는 관광 명소로서의 인기를 끌었고, 최근에는 다양한 조명과 이벤트를 통해 더욱 많은 방문객을 유치하고 있습니다.

특징:
도쿄 타워는 독특한 디자인과 화려한 조명으로 유명합니다. 특히, 야경이 아름다워 저녁 시간에 방문하면 환상적인 경관을 감상할 수 있습니다. 타워 내부에는 전망대가 두 개 있으며, 첫 번째 전망대는 150미터, 두 번째 전망대는 250미터에 위치해 있어 도쿄 시내를 한눈에 내려다볼 수 있습니다. 또한, 타워 주변에는 다양한 상점과 레스토랑이 있어 관광과 쇼핑을 동시에 즐길 수 있는 장소입니다.

방문 팁:
도쿄 타워를 방문할 때는 미리 온라인으로 티켓을 구매하는 것이 좋습니다. 특히 주말이나 공휴일에는 많은 인파가 몰리기 때문에 대기 시간을 줄일 수 있습니다. 또한, 일몰 시간에 맞춰 방문하면 낮과 밤의 경치를 모두 즐길 수 있습니다. 주변의 도쿄 타워 공원에서 산책을 하거나, 인근의 아카바네 공원에서 여유로운 시간을 보내는 것도 추천합니다. 마지막으로, 타워의 조명 쇼가 진행되는 시간에 맞춰 방문하면 더욱 특별한 경험을 할 수 있습니다.


### 4-2. 각 단계 결과를 모두 확인하는 연결

`RunnablePassthrough.assign()`은 입력 딕셔너리를 그대로 유지하면서 새 키를 추가합니다.

```
{"destination": "로마"}
  → assign(attraction=chain1) → {"destination": "로마", "attraction": "콜로세움"}
  → assign(detail=chain2)     → {"destination": "로마", "attraction": "콜로세움", "detail": "..."}
```

In [9]:
travel_chain = (
    RunnablePassthrough.assign(attraction=chain1)   # 1단계 결과 → attraction
    | RunnablePassthrough.assign(detail=chain2)     # 2단계 결과 → detail
)

def run(destination):
    result = travel_chain.invoke({"destination": destination})
    print(f"입력: {result['destination']}")
    print(f"1단계 결과: {result['attraction']}")
    print("2단계 결과:")
    print(result["detail"])
    print("=" * 50)

run("로마")

입력: 로마
1단계 결과: 콜로세움
2단계 결과:
명소: 콜로세움

역사:
콜로세움은 이탈리아 로마에 위치한 고대 로마의 원형 경기장으로, 기원후 70년에서 80년 사이에 건설되었습니다. 당시 로마 제국의 황제 베스파시아누스에 의해 시작되어 그의 아들 티투스에 의해 완공되었습니다. 콜로세움은 로마의 상징적인 건축물 중 하나로, 최대 50,000명에서 80,000명까지 수용할 수 있는 규모를 자랑하며, 검투사 경기, 야생 동물 사냥, 전투 재현 등 다양한 오락이 펼쳐졌습니다. 이곳은 로마의 정치적, 사회적, 문화적 중심지로서 중요한 역할을 했습니다.

특징:
콜로세움은 고대 로마 건축의 걸작으로, 원형의 구조와 뛰어난 공학 기술이 돋보입니다. 외관은 화강암과 콘크리트로 이루어져 있으며, 80개의 출입구가 있어 관중들이 신속하게 입장하고 퇴장할 수 있도록 설계되었습니다. 내부에는 복잡한 통로와 계단이 있어 관중들이 편리하게 자리를 찾을 수 있게 되어 있습니다. 또한, 콜로세움의 지하에는 검투사와 동물들이 대기하던 공간이 있었으며, 이를 통해 당시의 오락 문화를 엿볼 수 있습니다. 현재는 유네스코 세계문화유산으로 지정되어 있으며, 로마의 대표적인 관광 명소로 많은 관광객들이 방문하고 있습니다.

방문 팁:
콜로세움을 방문할 때는 사전 예약을 권장합니다. 특히 성수기에는 긴 대기줄이 생길 수 있으므로 온라인으로 미리 티켓을 구매하는 것이 좋습니다. 또한, 가이드 투어를 이용하면 콜로세움의 역사와 건축적 특징에 대한 깊이 있는 설명을 들을 수 있어 더욱 풍부한 경험이 될 것입니다. 방문 시에는 편안한 신발을 착용하고, 날씨에 맞는 옷차림을 준비하는 것이 좋습니다. 내부는 넓고 복잡하므로 충분한 시간을 두고 여유롭게 관람하는 것을 추천합니다.


In [10]:
for city in ["뉴욕", "도쿄"]:
    run(city)

입력: 뉴욕
1단계 결과: 자유의 여신상
2단계 결과:
명소: 자유의 여신상

역사:
자유의 여신상은 1886년 프랑스가 미국 독립 100주년을 기념하여 선물한 조각상으로, 프레데리크 바르톨디가 디자인하고 구스타브 에펠이 내부 구조를 설계했습니다. 이 조각상은 자유와 민주주의의 상징으로, 뉴욕 항구에 위치하여 미국으로 이민 오는 사람들을 맞이하는 역할을 해왔습니다. 여신상은 46미터의 높이를 자랑하며, 그 위에 있는 대좌와 함께 총 93미터에 달하는 높이를 가지고 있습니다. 1984년에는 유네스코 세계문화유산으로 등록되었습니다.

특징:
자유의 여신상은 왼손에 독립선언서(1776년 7월 4일)를 상징하는 문서를 들고 있으며, 오른손에는 횃불을 높이 들어 자유의 빛을 상징합니다. 여신상의 머리에는 일곱 개의 뿔이 있어 일곱 대륙과 바다를 나타내고 있습니다. 조각상의 외부는 구리로 만들어졌으며, 시간이 지나면서 산화되어 현재의 녹색을 띠게 되었습니다. 또한, 여신상의 내부에는 관람객들이 올라갈 수 있는 계단이 있어, 전망대에서 뉴욕 시내와 항구의 멋진 경관을 감상할 수 있습니다.

방문 팁:
자유의 여신상을 방문할 때는 미리 온라인으로 티켓을 구매하는 것이 좋습니다. 특히, 여름철에는 많은 관광객이 몰리기 때문에 사전 예약이 필수입니다. 페리를 타고 리버티 섬에 도착하면, 여신상과 함께 엘리스 섬의 이민 박물관도 함께 방문할 수 있습니다. 또한, 여신상에 올라가고 싶다면 'Crown Ticket'을 선택해야 하며, 이는 수량이 제한되어 있으므로 미리 예약하는 것이 중요합니다. 방문 시에는 편안한 신발을 착용하고, 날씨에 맞는 옷을 준비하는 것이 좋습니다.
입력: 도쿄
1단계 결과: 도쿄 타워
2단계 결과:
명소: 도쿄 타워  
역사:  
도쿄 타워는 1958년에 완공된 일본의 상징적인 구조물로, 높이는 333미터입니다. 이 타워는 프랑스 파리의 에펠탑에서 영감을 받아 설계되었으며, 일본의 경제 성장과 기술 발전을 상징하는 건축물로 자리 잡았습니다. 처음에는 방송 